# FloodAI v0.1 — Flood Occurrence Pipeline (Driver Notebook)

**This notebook contains no modeling logic.** It only orchestrates calls into
the `floodai` package under `src/`. If you find yourself wanting to edit a
formula, threshold, or feature definition here — stop, and edit the
corresponding module in `src/floodai/` instead, then re-run this notebook.
That separation is what makes the project testable and reviewable.

## Before running

1. This pipeline reports **held-out metrics only**. There is no code path in
   this package that can produce a "nicer" training-inclusive number labeled
   as a headline result — `evaluation/metrics.py` enforces this at the type
   level, not by convention.
2. The IMD data provider has not been executed against the live IMD server
   from the environment this code was authored in. **Run
   `notebooks/validate_first_run.py` first** and read its output before
   trusting any downstream number in this notebook.
3. Three basins are studied: Ganga (Bihar), Brahmaputra (Assam), Mahanadi
   (Odisha). Mahanadi is included specifically as a hard case — if it
   produces near-zero F1, that is a reportable finding (consistent with the
   reservoir/cyclone-driven flood mechanism documented in `config/config.yaml`),
   not a bug to hide.


In [ ]:
# Setup — run once per Colab session
import os
if not os.path.exists("requirements.txt"):
    !git clone https://github.com/souga15/FloodAI.git
    %cd FloodAI
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q imdlib


In [ ]:
import sys
sys.path.insert(0, "src")

import logging
from pathlib import Path

from floodai.config import load_config
from floodai.logging_utils import setup_logging, write_run_manifest

CONFIG_PATH = Path("config/config.yaml")
cfg = load_config(CONFIG_PATH)
logger = setup_logging(cfg.output_dir, run_name=cfg.experiment_id)
manifest_path = write_run_manifest(cfg.output_dir, cfg.experiment_id, CONFIG_PATH, cfg.random_seed)
logger.info("Run manifest written to %s", manifest_path)


## Step 0 — Validate data source before proceeding

**Do not skip this.** See the docstring in `validate_first_run.py` for why.


In [ ]:
!python notebooks/validate_first_run.py
# If this prints [FAIL] anywhere above, STOP and resolve before continuing.


## Step 1 — Generate basin points (reproducible, not hand-picked)

In [ ]:
from floodai.gis.points import generate_grid_fallback_points, basin_points_to_dataframe

# NOTE: using the deterministic grid fallback here because no administrative
# boundary shapefile is bundled with this package (geopandas + a GADM/OSM
# India boundaries file is required for the preferred admin-centroid method —
# see floodai.gis.points.generate_admin_centroid_points). If you have access
# to an India district/sub-district boundary file, swap this call for that
# function and re-run — it is a one-line change, by design.
all_points = []
for basin_key, basin_cfg in cfg.basins.items():
    pts = generate_grid_fallback_points(
        basin_key=basin_key,
        bbox=basin_cfg.bbox,
        n_points_target=basin_cfg.n_points_target,
        seed=cfg.random_seed,
    )
    all_points.extend(pts)

points_df = basin_points_to_dataframe(all_points)
points_df.to_csv(cfg.output_dir / "basin_points.csv", index=False)
print(f"Generated {len(points_df)} points across {points_df['basin_key'].nunique()} basins")
points_df.groupby("basin_key").size()


## Step 2 — Collect IMD rainfall for every point

This is the long-running step (per-point, per-year fetches). Runtime will
depend on IMD server response and `imdlib`'s yearly-file caching — expect
this to take longer than the old NASA-POWER per-station loop, since IMD
distributes whole-country yearly grids rather than point queries.


In [ ]:
from floodai.data.rainfall_providers import get_rainfall_provider
import pandas as pd
import time

provider = get_rainfall_provider(
    cfg.raw["data_sources"]["rainfall"]["provider"],
)
start_year = cfg.raw["data_sources"]["rainfall"]["start_year"]
end_year = cfg.raw["data_sources"]["rainfall"]["end_year"]

all_series = []
for i, row in points_df.iterrows():
    try:
        df_point = provider.fetch_point_series(
            row["lat"], row["lon"], f"{start_year}0101", f"{end_year}1231"
        )
        df_point["point_id"] = row["point_id"]
        df_point["basin_key"] = row["basin_key"]
        all_series.append(df_point)
    except Exception as e:
        logger.error("Failed to fetch point %s: %s", row["point_id"], e)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(points_df)} points fetched...")

rainfall_df = pd.concat(all_series, ignore_index=True)
rainfall_df.to_parquet(cfg.output_dir / "rainfall_raw.parquet")
print(f"Collected {len(rainfall_df):,} point-days. Citation: {provider.citation()}")


## Step 3 — Flood event labelling

Loads `data/flood_events_basins.csv` — **this file does not exist yet and
must be created** (see `documentation/flood_events_schema.md` for the
required columns). Only CWC/DFO/EM-DAT sourced events are accepted per
`config.yaml`; "News"-only sourced events from the prior WPT dataset are
deliberately excluded here pending independent verification.


In [ ]:
flood_events_path = Path(cfg.raw["data_sources"]["flood_events"]["path"])
if not flood_events_path.exists():
    raise FileNotFoundError(
        f"{flood_events_path} not found. This file must be created with "
        f"verified flood events (CWC/DFO/EM-DAT sourced only — see "
        f"documentation/flood_events_schema.md). The prior project's "
        f"33-station event list cannot be reused as-is: this study uses "
        f"basin-distributed points, not the same station identifiers, and "
        f"News-only events are excluded per config.yaml."
    )

flood_events_df = pd.read_csv(flood_events_path, parse_dates=["Start", "End"])
allowed_sources = set(cfg.raw["data_sources"]["flood_events"]["sources_allowed"])
bad_rows = flood_events_df[~flood_events_df["Source"].isin(allowed_sources)]
if len(bad_rows) > 0:
    raise ValueError(
        f"{len(bad_rows)} flood events use a Source not in {allowed_sources}. "
        f"Fix data/flood_events_basins.csv before proceeding."
    )
print(f"Loaded {len(flood_events_df)} verified flood events")


## Step 4 — Feature engineering (leakage-safe; see tests/test_features.py)

In [ ]:
from floodai.features.pipeline import (
    add_temporal_features, add_rainfall_window_features,
    add_scs_cn_runoff, add_interaction_features,
)

df = rainfall_df.merge(points_df[["point_id", "lat", "lon", "basin_key"]], on=["point_id", "basin_key"])
df = df.sort_values(["point_id", "Date"]).reset_index(drop=True)
df = add_temporal_features(df)
df = add_rainfall_window_features(df, group_col="point_id")
# CN/elevation/humidity/temperature joins to df go here once Step 2's IMD
# fetch is confirmed working end-to-end and terrain covariates (gis/terrain.py)
# have been run against real SRTM tiles for these basins.
print(f"Feature matrix: {df.shape}")


## Step 5+ — Labelling, split, SMOTE, tuning, evaluation, LOBO

These steps call directly into `floodai.training.*` and
`floodai.evaluation.metrics` exactly as exercised in
`tests/test_integration.py`. They are intentionally left as a TODO scaffold
here rather than filled in against unverified real data — fill these in only
after Steps 0–4 above have been run once in your Colab environment and their
outputs spot-checked (per `validate_first_run.py` and the missing/plausible
checks logged by `floodai.data.base.validate_daily_series`).


In [ ]:
# Steps 5-12: Full pipeline â€” CORRECTED to use floodai package modules
#
# This replaces a version of this cell that bypassed the floodai package
# entirely and reimplemented SMOTE/training inline. That version had two
# real bugs, both now structurally prevented:
#
#   1. `Year` (a raw integer column from add_temporal_features) was left in
#      feature_cols. With only ~10 verified flood events clustered in
#      certain years, the model could use Year as a lookup key for "which
#      years have a labeled flood" instead of learning rainfall/terrain
#      patterns. This produced LOBO AUC=1.000 / Recall=1.000 identically
#      across three basins with different flood mechanisms -- a sign of
#      leakage, not generalization.
#   2. `SMOTE(random_state=cfg.random_seed)` was called with no
#      `sampling_strategy`, silently defaulting to 50/50 class balancing
#      instead of the ~10% specified in config.yaml.
#
# Both are now caught automatically: select_model_features() excludes Year
# by construction, and resample_training_only() raises if the resulting
# ratio doesn't match what was requested.

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

from floodai.evaluation.metrics import DataProvenance, evaluate, report_headline
from floodai.features.governance import assert_no_forbidden_columns, select_model_features
from floodai.models.xgb_model import build_xgb_classifier, fit_with_validation
from floodai.training.imbalance import resample_training_only
from floodai.training.label_sufficiency import check_basin_has_positives, check_split_has_positives
from floodai.training.lobo import run_lobo_cv
from floodai.training.threshold import select_f1_optimal_threshold
from floodai.training.tuning import run_optuna_search

# â”€â”€ Step 5: Label floods â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def label_floods(df, flood_events_df):
    df = df.copy()
    df['Flood_Occurred'] = 0
    for _, ev in flood_events_df.iterrows():
        if 'basin_key' in flood_events_df.columns:
            mask = (
                (df['basin_key'] == ev['basin_key']) &
                (df['Date'] >= ev['Start']) &
                (df['Date'] <= ev['End'])
            )
        else:
            mask = (df['Date'] >= ev['Start']) & (df['Date'] <= ev['End'])
        df.loc[mask, 'Flood_Occurred'] = 1
    return df

df = label_floods(df, flood_events_df)
vc = df['Flood_Occurred'].value_counts()
print(f"Flood label distribution:\n{vc}")
print(f"Positive rate: {vc.get(1,0)/len(df)*100:.2f}%")

# â”€â”€ Step 5b: FAIL FAST if any split or basin lacks enough positive labels â”€â”€
# This is the check that would have caught "10 total events, none in the
# test window" immediately, instead of after a 35-minute Optuna run.
check_split_has_positives(
    df, date_col='Date', label_col='Flood_Occurred',
    train_years=cfg.raw['split']['train_years'],
    val_years=cfg.raw['split']['val_years'],
    test_years=cfg.raw['split']['test_years'],
    min_positives_per_split=5,
)
basin_counts = check_basin_has_positives(df, basin_col='basin_key', label_col='Flood_Occurred')
print(f"Per-basin positive counts: {basin_counts}")
print("[OK] All splits have sufficient positive labels. Proceeding.")

# â”€â”€ Step 6: Temporal train / val / test split â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
train_years = cfg.raw['split']['train_years']
val_years   = cfg.raw['split']['val_years']
test_years  = cfg.raw['split']['test_years']

df_train = df[df['Date'].dt.year.isin(train_years)].copy()
df_val   = df[df['Date'].dt.year.isin(val_years)].copy()
df_test  = df[df['Date'].dt.year.isin(test_years)].copy()

# CORRECTED: feature_cols now comes from the governed allowlist, not a
# manually maintained exclude set. This is what keeps Year (and any future
# leakage-risk column) out automatically.
feature_cols = select_model_features(df)
assert_no_forbidden_columns(feature_cols)
print(f"\nSelected {len(feature_cols)} governed features (Year excluded by design):")
print(feature_cols)

X_train, y_train = df_train[feature_cols].values, df_train['Flood_Occurred'].values
X_val,   y_val   = df_val[feature_cols].values,   df_val['Flood_Occurred'].values
X_test,  y_test  = df_test[feature_cols].values,  df_test['Flood_Occurred'].values

print(f"\nSplit sizes â€” Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# â”€â”€ Step 7: RobustScaler â€” fit on train ONLY â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)
print("RobustScaler fitted on train set only.")

# â”€â”€ Step 8: SMOTE on training set only â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# CORRECTED: uses resample_training_only(), which enforces the configured
# sampling_strategy and raises if the actual post-SMOTE ratio drifts from it.
imb_cfg = cfg.raw['imbalance']
X_train_res, y_train_res = resample_training_only(
    X_train_scaled, y_train,
    sampling_strategy=imb_cfg['sampling_strategy'],
    k_neighbors_max=imb_cfg['k_neighbors_max'],
    seed=cfg.random_seed,
)
print(f"After SMOTE â€” X_train_res: {X_train_res.shape}, "
      f"positives: {y_train_res.sum()} / {len(y_train_res)} "
      f"({y_train_res.mean():.2%}, target was {imb_cfg['sampling_strategy']/(1+imb_cfg['sampling_strategy']):.2%})")

# â”€â”€ Step 9: Optuna hyperparameter search (config-driven search space) â”€â”€â”€â”€â”€â”€
search_space = cfg.raw['model']['optuna']['search_space']
n_trials = cfg.raw['model']['optuna']['n_trials']
early_stopping_rounds = cfg.raw['model']['early_stopping_rounds']

best_params = run_optuna_search(
    X_train_res, y_train_res, X_val_scaled, y_val,
    search_space=search_space, n_trials=n_trials,
    early_stopping_rounds=early_stopping_rounds, seed=cfg.random_seed,
)
print(f"\nBest params: {best_params}")

best_model = build_xgb_classifier(best_params, early_stopping_rounds, cfg.random_seed)
best_model = fit_with_validation(best_model, X_train_res, y_train_res, X_val_scaled, y_val)

# â”€â”€ Step 10: F1-optimal threshold on validation set â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
val_proba = best_model.predict_proba(X_val_scaled)[:, 1]
tau_star = select_f1_optimal_threshold(y_val, val_proba)

# â”€â”€ Step 11: Evaluate on held-out test set (provenance-tagged) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
test_proba = best_model.predict_proba(X_test_scaled)[:, 1]
result = evaluate(
    y_test, test_proba, threshold=tau_star,
    set_name=f"test_{test_years[0]}_{test_years[-1]}",
    provenance=DataProvenance.HELD_OUT,
)
headlined = report_headline(result)  # raises if provenance isn't HELD_OUT â€” see evaluation/metrics.py

print("\n" + "="*55)
print("  HELD-OUT TEST SET RESULTS (headline-approved)")
print("="*55)
print(f"  ROC-AUC   : {headlined.roc_auc:.4f}")
print(f"  PR-AUC    : {headlined.pr_auc:.4f}")
print(f"  F1 Score  : {headlined.f1:.4f}")
print(f"  MCC       : {headlined.mcc:.4f}")
print(f"  Bal. Acc  : {headlined.balanced_accuracy:.4f}")
print(f"  FAR       : {headlined.far:.4f}")
print(f"  CSI       : {headlined.csi:.4f}")
print(f"  Threshold : {headlined.threshold:.4f}")
tn, fp, fn, tp = headlined.confusion
print(f"  TN={tn:>6}  FP={fp:>6}  FN={fn:>6}  TP={tp:>6}")
print("="*55)

# â”€â”€ Step 12: Leave-One-Basin-Out (LOBO) cross-validation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# CORRECTED: uses run_lobo_cv() from the package, which (a) calls
# assert_no_forbidden_columns() internally, and (b) routes all SMOTE calls
# through resample_training_only(), so the Year-leakage and SMOTE-ratio bugs
# cannot recur in this step either.
print("\nRunning LOBO cross-validation (governed)...")
lobo_results = run_lobo_cv(
    df, feature_columns=feature_cols, target_column='Flood_Occurred',
    basin_column='basin_key', best_params=best_params,
    early_stopping_rounds=early_stopping_rounds,
    smote_sampling_strategy=imb_cfg['sampling_strategy'],
    smote_k_neighbors_max=imb_cfg['k_neighbors_max'],
    seed=cfg.random_seed,
)

lobo_df = pd.DataFrame([{
    'held_out_basin': r.set_name.replace('LOBO_held_out_', ''),
    'ROC_AUC': r.roc_auc, 'F1': r.f1, 'MCC': r.mcc, 'n_positive': r.n_positive,
} for r in lobo_results])
print("\nLOBO Summary (provenance=loso_held_out â€” NOT a headline metric, see metrics.py):")
print(lobo_df.to_string(index=False))
if len(lobo_df) > 0:
    print(f"\nMean LOBO ROC-AUC : {lobo_df['ROC_AUC'].mean():.4f} +/- {lobo_df['ROC_AUC'].std():.4f}")
    print(f"Mean LOBO F1      : {lobo_df['F1'].mean():.4f} +/- {lobo_df['F1'].std():.4f}")
    print("\nNOTE: If ROC_AUC == 1.000 for every basin again, STOP and investigate")
    print("      before reporting -- that pattern previously indicated leakage,")
    print("      not genuine cross-basin generalization. Check feature_cols")
    print("      above for anything that could identify a specific basin/year")
    print("      rather than encode rainfall/terrain physics.")
